# 02 - Multivariate EDA, aimed at feature engineering

"01_eda.ipynb" looks at one feature at a time. This notebook asks the questions that feed feature engineering:

1. Signal: which features rank accepted above denied on their own, measured with the competition metric (AUC)?
2. Redundancy: which features carry the same information (keep one, merge, or build one combined flag)?
3. Shape: which transform does each numeric feature need (log, clip, bins, splines)?
4. Interactions: which pairs need a combined feature (ratio, crossed category)?
5. Structure: PCA on the full feature set. Is there low-dimensional structure?
6. Segments: clustering and a shallow tree. Do denials concentrate in a few segments?

Every feature-engineering decision, with where it comes from, is collected in "reports/feature_plan.md" (the spec for "src/features.py").

Conventions: every score is out-of-fold on the frozen folds ("data/processed/folds.parquet"), so it is measured the same way as the models' CV. Recommendations are marked "=> FE".

In [ ]:
import sys
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
from joblib import Parallel, delayed
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from scipy.cluster.hierarchy import fcluster, leaves_list, linkage
from scipy.spatial.distance import squareform
from scipy.stats import skew
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
from sklearn.metrics import log_loss, roc_auc_score, silhouette_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import SplineTransformer, StandardScaler
from sklearn.tree import DecisionTreeClassifier

sys.path.insert(0, str(Path.cwd().parent / "src"))  # notebook runs from notebooks/
from config import load_config
from data import CODED, FEATURES, NUMERIC, TARGET, load
from folds import load_folds

pd.set_option("display.max_columns", 60, "display.width", 200, "display.max_colwidth", None)

train, test = load()
train = train.merge(load_folds(load_config()), on="id", how="left", validate="1:1")
y, fold = train[TARGET].to_numpy(), train["fold"].to_numpy()
OVERALL = y.mean()

# Feature groups that matter for the two objectives (leaderboard AND fair model).
PROTECTED = [c for c in FEATURES if any(s in c for s in ("race", "ethnicity", "sex")) or c.endswith("_age")]
PROCESS = ["preapproval", "initially_payable_to_institution", "submission_of_application", "aus-1"]  # leakage suspects: process fields
assert len(PROTECTED) == 17
train.shape, test.shape

In [ ]:
# One style for every figure: recessive solid hairline grid, no top/right spines, titles left.
# Colours = validated colour-blind-safe categorical slots 1-3, always in this order.
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK, MUTED, GRID, SURFACE = "#52514e", "#898781", "#e1e0d9", "#fcfcfb"
# Sequential (magnitude, 0 -> 1): one hue, light to dark. Diverging (sign): blue <- grey -> red.
SEQ = LinearSegmentedColormap.from_list("seq", [SURFACE, "#cde2fb", "#6da7ec", "#2a78d6", "#184f95", "#0d366b"])
DIV = LinearSegmentedColormap.from_list("div", ["#184f95", "#6da7ec", "#f0efec", "#ec8a88", "#b3261e"])
plt.rcParams.update({
    "figure.dpi": 110, "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "font.size": 9, "axes.titlesize": 11, "axes.titlelocation": "left", "axes.titlepad": 10,
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#c3c2b7",
    "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "legend.frameon": False,
})

## 1. Which features carry signal on their own?

Measure: for each feature alone, fit a one-feature predictor on 4 folds, predict the 5th, compute the AUC. 0.5 = no signal, 1 = perfect ranking. Since it is out-of-fold, a feature can't score high by memorising (this matters for "county_code" with its 2,898 levels in train).

- Categorical => accept rate per level, shrunk towards the fold's overall rate: $(\text{accepted} + m \cdot \text{prior}) / (\text{rows} + m)$. A level needs $m$ rows before its own rate gets half the weight, and an unseen level gets the prior. $m = 20$ is the middle of the grid {5, 20, 100} planned for the target encoding. Its effect is checked right below, and it is small.
- Numeric => a decision tree on that one column, each leaf >= 1% of the fit rows (~840 rows, so a leaf's accept rate has SE ~0.012). A tree follows any shape (cliffs, plateaus, the non-monotone dip of "loan_amount"), sends NaN to whichever side fits better, and never splits tied values (81% of "loan_term" is exactly 360). A plain AUC of the raw value would miss non-monotone shapes.

In [ ]:
M_SMOOTH = 20       # pseudo-rows at the prior rate (see text above)
LEAF_SHARE = 0.01   # smallest tree leaf = 1% of the fit rows (see text above)


def oof_pred(x, m=M_SMOOTH):
    # Out-of-fold prediction of the target from the single column x.
    pred = np.empty(len(x))
    for k in np.unique(fold):
        fit, val = fold != k, fold == k
        if x.name in NUMERIC:
            tree = DecisionTreeClassifier(min_samples_leaf=LEAF_SHARE, random_state=0)
            tree.fit(x[fit].to_frame(), y[fit])
            pred[val] = tree.predict_proba(x[val].to_frame())[:, 1]
        else:
            code = x.cat.codes.to_numpy() + 1                  # 0 = missing, kept as its own level
            n_levels = len(x.cat.categories) + 1
            rows = np.bincount(code[fit], minlength=n_levels)
            accepted = np.bincount(code[fit], weights=y[fit], minlength=n_levels)
            prior = y[fit].mean()
            rate = (accepted + m * prior) / np.maximum(rows + m, 1)
            rate[rows == 0] = prior                            # level unseen in the fit folds
            pred[val] = rate[code[val]]
    return pred


def fold_auc(pred):
    # AUC on each held-out fold.
    return np.array([roc_auc_score(y[fold == k], pred[fold == k]) for k in np.unique(fold)])


per_fold = pd.DataFrame({c: fold_auc(oof_pred(train[c])) for c in FEATURES}).T
auc = pd.DataFrame({"auc": per_fold.mean(axis=1), "lo": per_fold.min(axis=1), "hi": per_fold.max(axis=1)})
auc = auc.sort_values("auc")

In [ ]:
group_color = [ORANGE if c in PROTECTED else AQUA if c in PROCESS else BLUE for c in auc.index]
pos = np.arange(len(auc))

fig, ax = plt.subplots(figsize=(9, 12))
ax.hlines(pos, auc.lo, auc.hi, color=group_color, lw=2)
ax.scatter(auc.auc, pos, c=group_color, s=42, zorder=3, edgecolor=SURFACE, linewidth=1.5)
ax.axvline(0.5, color=MUTED, lw=1)
ax.set_yticks(pos, auc.index)
ax.set_ylim(-0.8, len(auc) - 0.2)
ax.grid(axis="y", visible=False)
ax.set_xlabel("out-of-fold AUC of a one-feature predictor  (dot = mean of 5 folds, bar = fold min to max, 0.5 = no signal)")
ax.set_title("Signal carried by each feature on its own")
ax.legend(handles=[Line2D([], [], marker="o", ls="", color=col, label=lab) for col, lab in
                   [(BLUE, f"other ({len(FEATURES) - len(PROTECTED) - len(PROCESS)})"),
                    (ORANGE, f"protected attribute ({len(PROTECTED)})"),
                    (AQUA, f"process / leakage suspect ({len(PROCESS)})")]],
          loc="lower right")
fig.tight_layout()

Reading:

- Top tier, 0.68 to 0.73: "aus-1" and DTI, then the money trio, LTV and "loan_term". The strongest single signals are the underwriting inputs and whether an automated underwriting system was run at all.
- Manufactured-home block, 0.67: "construction_method", "derived_dwelling_category" and both "manufactured_home_\*" fields score almost the same.
- Geography, 0.61 to 0.63: county, MSA, state and MSA median income are close. County, with 2,898 levels, is barely above state (54 levels): at this sample size most of what the fine levels add is noise.
- Protected attributes, 0.53 to 0.57: race ~0.57, sex ~0.56, ethnicity ~0.55, age ~0.53. Weaker than the financial variables, clearly above 0.5.
- Co-applicant columns, all ~0.55: "no co-applicant" is the same ~57.5k rows in every one of them (sex = 5, race = 8, ethnicity = 5, observed = 4, age blank). Their signal is mostly "is there a co-applicant", not demographics. The "applicant_\*_observed" fields have none (~0.50).
- Around 0.50: "total_units", "conforming_loan_limit", "occupancy_type", "lien_status" and the rare yes/no loan flags.
- Caveat: "preapproval" scores only 0.53 although "requested" is accepted 100% of the time. AUC measures the whole ranking, and a perfect rule that covers 4.5% of rows moves it little. This ranking does not replace a level-by-level leakage scan.

=> FE
- Build **has_coapplicant** (one flag). It keeps the co-applicant signal when the fair model drops the 17 demographic columns. Its own fairness status (proxy for marital status / joint applications) must be checked in the fairness analysis.
- No feature is dropped for a low single-feature AUC. Trees ignore useless columns at little cost, and a weak feature can matter in an interaction (e.g. "occupancy_type" x LTV). The group around 0.50 is the first place to prune if the linear model needs fewer columns.
- High-cardinality geography needs **shrinkage** (smoothed out-of-fold target encoding).

### 1b. Does the smoothing strength m change the picture for geography?

Only the high-cardinality columns are sensitive to $m$. Every tree leaf and almost every level of the other categoricals has hundreds of rows, far more than $m$.

In [ ]:
geo = ["county_code", "derived_msa-md", "state_code"]
pd.DataFrame({m: [fold_auc(oof_pred(train[c], m)).mean() for c in geo] for m in [0, 5, 20, 100]},
             index=geo).rename_axis(columns="m").round(4)

County gains ~0.007 AUC from no smoothing (m = 0) to m = 20, then loses a little at m = 100. MSA and state barely move (their levels are large). So the ranking above does not depend on the choice of m.

=> FE: out-of-fold target encoding for "county_code" with **m of order 20**. The feature stage keeps the grid {5, 20, 100} and the CV decides. For state and MSA, plain level rates are enough.

## 2. Which features carry the same information?

Measure: bias-corrected Cramer's V (Bergsma 2013) for every pair of features. 0 = independent, 1 = one column determines the other. It works for any pair of types once numerics are cut into 20 equal-count bins (5% of rows each; a value that fills a whole bin on its own, like "loan_term" = 360, gets its own bin; NaN is its own level). The correction removes the part of V that pure chance produces in large sparse tables (county x anything).

Rows carrying any "1111" (3,016) are left out here: "1111" is one row-level fact (found in 01_eda). Since it is shared by 14 columns, it alone gives V ~0.7 between all of them and hides everything else.

Features are ordered by hierarchical clustering (average linkage on 1 - V). The boxes are the groups obtained by cutting the tree at 1 - V = 0.3, i.e. groups whose average V >= 0.7. Why 0.7: $V^2$ is the share of the maximum possible association, and $0.7^2 \approx 0.5$.

In [ ]:
def discretize(x, q=20):
    # Equal-count bins; a value that fills a whole bin on its own (loan_term = 360) gets its own bin.
    edges = np.nanquantile(x, np.linspace(0, 1, q + 1))
    mass = np.unique(edges[1:][np.diff(edges) == 0])
    edges = np.unique(np.concatenate([edges, np.nextafter(mass, -np.inf)]))
    return pd.cut(x, edges, include_lowest=True)


def cramers_v(a, b):
    # Bias-corrected Cramer's V (Bergsma 2013) between two integer-coded columns.
    r, k = a.max() + 1, b.max() + 1
    t = np.bincount(a * k + b, minlength=r * k).reshape(r, k)
    t = t[t.sum(1) > 0][:, t.sum(0) > 0]                        # drop empty levels
    n, (r, k) = t.sum(), t.shape
    expected = np.outer(t.sum(1), t.sum(0)) / n
    phi2 = ((t - expected) ** 2 / expected).sum() / n
    phi2 = max(0.0, phi2 - (k - 1) * (r - 1) / (n - 1))        # remove what chance alone produces
    r, k = r - (r - 1) ** 2 / (n - 1), k - (k - 1) ** 2 / (n - 1)
    return np.sqrt(phi2 / min(k - 1, r - 1))


EXEMPT = (train[CODED].astype("string") == "1111").any(axis=1).to_numpy()
codes = {c: (discretize(train[c]) if c in NUMERIC else train[c]).cat.codes.to_numpy()[~EXEMPT].astype(np.int64) + 1
         for c in FEATURES}                                        # 0 = missing, its own level
V = pd.DataFrame(1.0, index=FEATURES, columns=FEATURES)
for i, a in enumerate(FEATURES):
    for b in FEATURES[i + 1:]:
        V.loc[a, b] = V.loc[b, a] = cramers_v(codes[a], codes[b])
print(f"{(~EXEMPT).sum():,} rows used, {EXEMPT.sum():,} rows with a 1111 code left out")

In [ ]:
GROUP_CUT = 0.3   # groups = average V >= 0.7 (see text above)
links = linkage(squareform(1 - V.to_numpy(), checks=False), method="average")
order = V.index[leaves_list(links)]
group = pd.Series(fcluster(links, t=GROUP_CUT, criterion="distance"), index=V.index)[order]
sizes = group.groupby(group, sort=False).size()          # groups are contiguous in the leaf order

fig, ax = plt.subplots(figsize=(11, 10))
im = ax.imshow(V.loc[order, order], cmap=SEQ, vmin=0, vmax=1)
start = 0
for size in sizes:
    if size > 1:
        ax.add_patch(Rectangle((start - 0.5, start - 0.5), size, size, fill=False, edgecolor=ORANGE, lw=1.5))
    start += size
ax.set_xticks(range(len(order)), order, rotation=90, fontsize=7)
ax.set_yticks(range(len(order)), order, fontsize=7)
ax.grid(False)
ax.spines[["left", "bottom"]].set_visible(False)
fig.colorbar(im, ax=ax, shrink=0.6, label="bias-corrected Cramer's V")
ax.set_title("Association between every pair of features (boxes: groups with average V >= 0.7)")
fig.tight_layout()

In [ ]:
rows = []
for g, members in group.groupby(group, sort=False):
    if len(members) > 1:
        m = list(members.index)
        pair_v = V.loc[m, m].to_numpy()[np.triu_indices(len(m), 1)]
        rows.append((", ".join(m), pair_v.min(), pair_v.mean()))
pd.DataFrame(rows, columns=["group", "min V", "mean V"]).round(2)

Reading, group by group (each one checked with a crosstab):

- "loan_type" + "derived_loan_product_type" (V = 1), and "lien_status" also has V = 1 with the derived column: "derived_loan_product_type" is exactly loan_type x lien_status, a one-to-one relabelling.
- "construction_method" + "derived_dwelling_category" + both "manufactured_home_\*" (V >= 0.99 with construction). "derived_dwelling_category" is exactly construction x (units >= 5). But the "manufactured_home_\*" fields are not duplicates. For site-built homes they are constant ("not applicable"), and inside the manufactured segment they split it strongly: "home + land" is accepted 0.52 against 0.26 for "home only" (5,638 rows), and direct land ownership 0.43 against 0.20 to 0.29 for leasehold / indirect.
- Geography (county, MSA, state, MSA median income): county determines state (V = 0.98), MSA (0.99) and the MSA median income (0.99), which is one number per MSA. It is a nested hierarchy.
- "total_units" + "conforming_loan_limit" (0.71): this comes from one shared rare level, not general association. Both are missing on the same 437 rows, and "derived_dwelling_category" labels those rows Multifamily. So a missing unit count means 5+ units, not "unknown".
- "business_or_commercial_purpose" + "occupancy_type" (0.84): 96% of business-purpose loans are investment properties. At a looser cut, DTI joins them through its missing level (explained just below: entity applicants).
- Demographics: "applicant_race-1" and "derived_race" (0.82), ethnicity (0.86), sex (0.74). The three "applicant_\*_observed" fields are one fact (0.97), and so are the three co-applicant ones (0.97).

Outside the boxes:
- "loan_term" vs "construction_method" 0.71: 300-month terms are manufactured homes.
- "aus-1" vs "loan_type" 0.66: TOTAL is the FHA system, GUS the USDA one.
- LTV vs "loan_type" 0.57: each loan program has its own standard LTV. 58% of FHA loans sit exactly at 96.5 (the 3.5% minimum down payment), 70% of VA loans at 100, and conventional loans cluster at 80 (20%).
- "loan_amount" vs "lien_status" 0.58: subordinate liens are small loans (median 15k vs 295k).

=> FE
- Trees: **drop "derived_loan_product_type" and "derived_dwelling_category"**. They are exact functions of other columns, and duplicates only split the importance. For the linear model the derived columns are useful, since they are ready-made crossed categories.
- **"total_units" missing => "5+"** (an ordered level), not NaN. Same rows => "conforming_loan_limit" missing = "not applicable (multifamily)".
- Keep both "manufactured_home_\*" fields. For the linear model they only act inside the manufactured segment, so encode them as **construction x secured type**.
- County encoding: **shrink towards the MSA / state rate** rather than the global rate (hierarchical target encoding). A small county then borrows from its neighbours, not from the whole country.
- Fair model: the demographic blocks must be removed together (already noted in 01_eda). One "observed" column would carry all the information of the three.

### 2b. Numeric features: monotone links

Cramer's V is strict for numerics: it reaches 1 only when one column determines the other at the bin level. Two columns with Spearman 0.87 (loan amount and property value) only reach V ~0.45. So for the 13 numeric columns here is Spearman's $\rho$, which measures monotone association (rank-based, so no transform is needed). Same ordering method, and only $|\rho| \geq 0.3$ is written.

In [ ]:
rho = train[NUMERIC].corr(method="spearman")
o = rho.index[leaves_list(linkage(squareform(1 - rho.abs().to_numpy(), checks=False), method="average"))]
rho = rho.loc[o, o]

fig, ax = plt.subplots(figsize=(8.5, 7))
im = ax.imshow(rho.mask(np.eye(len(o), dtype=bool)), cmap=DIV, vmin=-1, vmax=1)   # diagonal (= 1) left blank
for i in range(len(o)):
    for j in range(len(o)):
        if i != j and abs(rho.iloc[i, j]) >= 0.3:
            ax.text(j, i, f"{rho.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7,
                    color="white" if abs(rho.iloc[i, j]) > 0.6 else "#0b0b0b")
ax.set_xticks(range(len(o)), o, rotation=90, fontsize=8)
ax.set_yticks(range(len(o)), o, fontsize=8)
ax.grid(False)
ax.spines[["left", "bottom"]].set_visible(False)
fig.colorbar(im, ax=ax, shrink=0.7, label="Spearman rho")
ax.set_title("Monotone association between numeric features")
fig.tight_layout()

Two blocks:

- Money: loan and property 0.87, property and income 0.70, loan and income 0.66. Property value also follows the area's income level (0.44 with the MSA median income, 0.39 with the tract's relative income).
- Tract size: population, owner-occupied units and 1-4 family homes (0.72 to 0.85), as in 01_eda.

Everything else is weak ($|\rho| < 0.3$). LTV has $\rho = -0.38$ with property value and almost nothing with loan amount.

=> FE
- The money trio is mostly one "size of the deal" axis. What differs between applicants is their proportions: **loan / income, property / income** (and loan / property = computed LTV). The ratios take the common size out and keep what each column adds. Whether they carry signal beyond the raw columns is tested with the interaction maps below.
- **Property value relative to the area**: property / MSA median income, a price level corrected for the local market.
- Tract size trio: for the linear model **keep one size column + shares** (owner-occupied / 1-4 family homes), as in 01_eda.

### 2c. A second row-level state: the applicant is not a natural person

In the V matrix, "balloon_payment" has V ~0.6 with applicant sex, race, ethnicity and age, which makes no sense for people. The link is one shared level: "applicant_sex" = 4, "not applicable", which HMDA uses when the applicant is not a natural person (a company or a trust).

In [ ]:
s = lambda c: train[c].astype("string")
entity = s("applicant_sex") == "4"
checks = pd.DataFrame({
    "race-1 = 7 (not applicable)": s("applicant_race-1") == "7",
    "ethnicity-1 = 4 (not applicable)": s("applicant_ethnicity-1") == "4",
    "age missing": train.applicant_age.isna(),
    "income missing": train.income.isna(),
    "DTI missing": train.debt_to_income_ratio.isna(),
    "investment property": s("occupancy_type") == "3",
    "business purpose": s("business_or_commercial_purpose") == "1",
    "balloon payment": s("balloon_payment") == "1",
    "accepted": train[TARGET] == 1,
})
print(f"sex = 4: {entity.sum():,} train rows, {(test.applicant_sex.astype('string') == '4').sum():,} test rows")
pd.DataFrame({"share if sex = 4": checks[entity].mean(), "share otherwise": checks[~entity].mean(),
              "share of all such rows that have sex = 4": [entity[checks[c]].mean() for c in checks]}).round(3)

Confirmed: 2,910 train rows (1,285 in test) form one coherent block. They have no person-level fields (race / ethnicity "not applicable" 99.9%, age and income missing ~100%, DTI missing 99%), and they are investment properties (98%) and mostly business loans (79%). They are accepted at 0.88, about the overall rate.

This block explains most of the "income, age and DTI missing together" block found in 01_eda: it accounts for 94% of the missing ages, 64% of the missing incomes and 38% of the missing DTIs.

=> FE
- **is_entity_applicant** flag (applicant_sex = 4). Income / age / DTI missing on these rows means "does not apply", not "unknown".
- Income and DTI missing outside this block (~1.7k and ~4.8k rows) is a different kind of missing. Keep a separate indicator for it.
- Fairness: these rows have no protected attributes at all. The fairness metrics must be computed on natural persons only, and this flag says who they are.

## 3. Which transform does each numeric feature need?

Trees only use the order of the values, so transforms don't change them. This section is for the linear model and any neural net: which encoding lets a linear term capture the feature?

The table below lists the facts that force a choice (sentinels, mass points, skew, outliers). The test after it measures the choice.

"total_units" is left out: 4 values, 97% of them = 1, so it is treated as an ordered category (with "5+" for missing). DTI and the ages are text categories. 01_eda showed the U-shape of DTI, so the linear model gets its levels one-hot rather than a single number.

In [ ]:
NUM = [c for c in NUMERIC if c != "total_units"]


def facts(x):
    top = x.value_counts(normalize=True)                  # shares among non-missing values
    return {"missing %": 100 * x.isna().mean(), "<= 0 %": 100 * (x <= 0).mean(),
            "most common": top.index[0], "its share %": 100 * top.iloc[0],
            "skew": skew(x.dropna()), "skew of log": skew(np.log(x[x > 0])),
            "median": x.median(), "p99.9": x.quantile(0.999), "max": x.max()}


pd.DataFrame({c: facts(train[c]) for c in NUM}).T.round(2)

- Tails: the money trio and LTV have extreme right tails (skew 98 to 305; the max loan is 375M, the max LTV $6 \cdot 10^8$). A linear model can't use them raw. Log removes the tail of the money trio (skew of log -1.2 to 0.4). For LTV it over-corrects (-3.5): its bulk is packed between 80 and 100.
- Sentinels: the census columns have 1.3 to 6.2% zeros (mostly the location-unknown block found in 01_eda), income 0.9% <= 0, housing age 6.2% zeros.
- Mass points: "loan_term" = 360 for 84% of the known values, LTV = 80 for 14%.
- Once the zeros are set aside, the census columns are only mildly skewed (0.1 to 2.2), and log does not make them more symmetric (-0.2 to -2.1).

### 3b. How much of a feature's signal does one linear term capture?

For each numeric feature alone: an out-of-fold logistic regression under 5 encodings, next to the one-feature tree used for the signal ranking (which follows any shape):

- flags only: a missing indicator and a "<= 0" indicator, no value
- flags + raw value
- flags + log(value)
- flags + rank (share of fit rows below the value, 0 to 1, insensitive to outliers)
- flags + cubic spline on log(value), knots at min / quartiles / max (7 basis functions)

In every variant, missing and <= 0 values are replaced by the median of the positive values and carried by their own indicator. 01_eda showed these rows behave as groups of their own (census zeros = location unknown, income <= 0 accepted 0.58 or 0.00).

Score: deviance explained (McFadden's $R^2$) = 1 - log loss(model) / log loss(constant), out-of-fold, in %. AUC can't be used here: a monotone transform of a single feature does not change its AUC, so raw and log would always tie.

Choices: quartile knots give each spline piece a quarter of the rows (>= 20k), enough to follow a plateau or a dip without chasing noise. The logistic regression keeps sklearn's default L2 penalty (C = 1): with 84k rows and <= 9 coefficients it has no visible effect.

In [ ]:
def encode(x_fit, x, kind):
    # Missing and <= 0 -> median of the positive fit values, each kept by its own indicator.
    sentinel = np.column_stack([np.isnan(x), x <= 0])
    pos_fit = x_fit[x_fit > 0]
    v = np.where(sentinel.any(axis=1), np.median(pos_fit), x)
    if kind == "flags only":
        return sentinel
    if kind == "raw":
        f = v[:, None]
    elif kind == "log":
        f = np.log(v)[:, None]
    elif kind == "rank":
        s = np.sort(pos_fit)
        f = ((np.searchsorted(s, v, "left") + np.searchsorted(s, v, "right")) / (2 * len(s)))[:, None]
    elif kind == "spline":
        knots = np.unique(np.quantile(np.log(pos_fit), [0, 0.25, 0.5, 0.75, 1]))
        f = SplineTransformer(knots=knots[:, None], degree=3).fit_transform(np.log(v)[:, None])
    return np.column_stack([f, sentinel])


def deviance_explained(p, k):
    # 1 - logloss(p) / logloss(fit-fold rate), on held-out fold k, in %.
    val = fold == k
    constant = np.full(val.sum(), y[fold != k].mean())
    return 100 * (1 - log_loss(y[val], p) / log_loss(y[val], constant))


KINDS = ["flags only", "raw", "log", "rank", "spline"]
linear = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))   # max_iter: convergence only
scores = {}
for c in NUM:
    x = train[c].to_numpy()
    s = {kind: [] for kind in KINDS + ["one-feature tree"]}
    for k in np.unique(fold):
        fit, val = fold != k, fold == k
        for kind in KINDS:
            X = encode(x[fit], x, kind)
            s[kind].append(deviance_explained(linear.fit(X[fit], y[fit]).predict_proba(X[val])[:, 1], k))
        tree = DecisionTreeClassifier(min_samples_leaf=LEAF_SHARE, random_state=0).fit(x[fit, None], y[fit])
        s["one-feature tree"].append(deviance_explained(tree.predict_proba(x[val, None])[:, 1], k))
    scores[c] = {kind: np.mean(v) for kind, v in s.items()}
shape = pd.DataFrame(scores).T.sort_values("one-feature tree", ascending=False)

In [ ]:
share = shape.div(shape.max(axis=1), axis=0).clip(lower=0)    # 1 = as good as the best encoding in the row

fig, ax = plt.subplots(figsize=(8.5, 5.5))
ax.imshow(share, cmap=SEQ, vmin=0, vmax=1, aspect="auto")
for i in range(shape.shape[0]):
    for j in range(shape.shape[1]):
        ax.text(j, i, f"{shape.iloc[i, j]:.1f}".replace("-0.0", "0.0"), ha="center", va="center", fontsize=8,
                color="white" if share.iloc[i, j] > 0.6 else "#0b0b0b")
ax.set_xticks(range(shape.shape[1]), shape.columns)
ax.xaxis.tick_top()
ax.set_yticks(range(len(shape)), shape.index)
ax.tick_params(length=0)
ax.grid(False)
ax.spines[:].set_visible(False)
ax.set_title("Deviance explained (%, out-of-fold) by one feature under each encoding.\n"
             "Colour = share of the best encoding in the row.", pad=28)
fig.tight_layout()

Five patterns:

1. Census block (tract population, owner-occupied units, 1-4 family homes, tract income %, minority %): the flags alone explain 4.7 to 5.6 points, where the best encoding reaches 5.4 to 6.1. The signal is the zero block (location unknown), and after it the value adds at most ~0.7 points in any form. Log, raw or rank make no difference.
2. Money trio: raw is useless (1.2 to 2.5, the outliers crush the scale), log recovers a part (3.1 to 7.8), and the spline on log reaches the tree (loan 7.2 vs 7.5, property 10.1 vs 10.2, income 8.1 vs 8.1). Log alone is furthest behind for "loan_amount", because of its dip around 85k (seen in 01_eda).
3. LTV: the missing flag alone gives 8.7 of the tree's 10.1, and a linear term on the value adds <= 0.5. Checked separately: one-hot 5-point bins (as in 01_eda) reach 9.6, and adding flags for the exact standard values (80, 90, 95, 96.5, 97, 100) reaches 10.0, the tree's level. Rows exactly at 80 are accepted 0.92 against 0.85 within 2 points of it; 90: 0.92 vs 0.86; 97: 0.92 vs 0.88; 100: 0.89 vs 0.85 (96.5: no difference). Hypothesis: a round LTV means a loan sized to a program standard, i.e. a complete, standard file.
4. "loan_term": every linear form stays <= 7.2 while the tree reaches 14.5, the highest of all numerics. The signal sits in a few exact values. Checked: 7 categories (360, 300, 240, 180, 120, other, missing) reach 13.5. 300 months are accepted 0.28, 240 months 0.54, 360 months 0.90 (mostly the manufactured segment, as in 01_eda).
5. MSA median income: the value adds ~1.2 to the flags, the tree ~0.7 more. That extra is not a shape: there is one value per MSA, so the tree is partly learning MSA identity, which the MSA encoding already covers. Housing age: the flags give 1.95 of 2.3, but that is the location-unknown block again. Outside it, housing age 0 (4,390 rows) is accepted 0.88, about the overall rate (checked in "reports/feature_plan.md").

=> FE (linear model; trees take the raw columns + the flags)
- Census block: **zeros become NaN + one "location unknown" flag**. Values raw, or log for the three size columns.
- Money trio: **cubic spline on log**, knots at the quartiles, plus the **income <= 0** flag.
- LTV: **missing flag**, **5-point bins** (one-hot) instead of a numeric term, and an **"LTV at a standard value"** flag (80, 90, 95, 97, 100). For trees the flag is optional (a tree can isolate a frequent exact value by itself), and the CV decides.
- "loan_term": **category** (360, 300, 240, 180, 120, other, missing), which covers 92% of rows by name.
- Housing age: nothing beyond the location-unknown flag. A zero flag is optional, CV decides.
- General rule: **every sentinel gets its own indicator before any transform**. Without the indicators, the "best transform" is decided by a few thousand sentinel rows.

## 4. Which features interact?

Two features interact when the effect of one depends on the value of the other. A linear model only sees an interaction if a crossed feature is built for it. Trees find interactions by themselves, but they cut along one axis at a time, so a boundary along a diagonal (a ratio) costs them many splits.

- 4a: maps of the money pairs. Does acceptance follow a ratio?
- 4b: does each candidate ratio add signal beyond its two columns?
- 4c: a scan of all 66 crossings of the 12 strongest features.
- 4d: the strongest crossings, read as tables.

### 4a. Money pairs: does acceptance follow a ratio?

Each hexagon is coloured by the accept rate of its rows, on the log-odds scale relative to the overall rate. Log-odds is the linear model's scale: there, "no interaction" means the effect of x is the same at every y. The rate uses a +0.5 continuity correction, so a cell with 0% or 100% stays finite. Hexagons with < 50 rows are left blank (a rate on 50 rows has SE <= 0.07). The axes are log scales over the central 99% of each variable.

The grey diagonals are lines of constant ratio. If the colour bands run parallel to them, the ratio is what matters. Vertical bands mean only x matters, horizontal bands only y.

In [ ]:
dollars = pd.DataFrame({"loan": train.loan_amount, "property": train.property_value,
                        "income": train.income * 1000,                     # income is in $k in the raw data
                        "MSA income": train.ffiec_msa_md_median_family_income})
LOGIT_OVERALL = np.log(OVERALL / (1 - OVERALL))
TICKS = np.log10([5e3, 1e4, 2e4, 5e4, 1e5, 2e5, 5e5, 1e6, 2e6, 5e6])
dollar_fmt = plt.FuncFormatter(lambda v, _: f"{10 ** v / 1e6:g}M" if v >= 6 else f"{10 ** v / 1e3:g}k")


def rel_logit(v):
    # Log-odds of acceptance in a cell (+0.5 continuity correction), relative to the overall log-odds.
    k, n = np.sum(v), len(v)
    return np.log((k + 0.5) / (n - k + 0.5)) - LOGIT_OVERALL


def rate_colorbar(fig, mappable, ax):
    # Colour is spaced in log-odds, ticks are written as accept rates.
    cb = fig.colorbar(mappable, ax=ax, shrink=0.5)
    t = np.arange(-2, 3)
    cb.set_ticks(t, labels=[f"{1 / (1 + np.exp(-(LOGIT_OVERALL + v))):.2f}" for v in t])
    cb.set_label(f"accept rate (grey = overall {OVERALL:.2f})")


MAPS = [("income", "loan", [1, 2, 3, 5]), ("income", "property", [2, 3, 5, 8]),
        ("property", "loan", [0.5, 0.8, 0.95]), ("MSA income", "property", [2, 3, 5, 8])]
fig, axes = plt.subplots(2, 2, figsize=(12, 10), layout="constrained")
for ax, (xc, yc, ratios) in zip(axes.flat, MAPS):
    ok = (dollars[xc] > 0) & (dollars[yc] > 0)
    lx, ly = np.log10(dollars[xc][ok]), np.log10(dollars[yc][ok])
    ext = [lx.quantile(0.005), lx.quantile(0.995), ly.quantile(0.005), ly.quantile(0.995)]
    hb = ax.hexbin(lx, ly, C=y[ok], reduce_C_function=rel_logit, gridsize=35, extent=ext, mincnt=50,
                   cmap=DIV.reversed(), vmin=-2, vmax=2, linewidths=0.3, edgecolors=SURFACE)
    for r in ratios:
        ax.axline((0, np.log10(r)), slope=1, color=INK, lw=0.7)          # y = r * x on log axes
    ax.set_xticks(TICKS)
    ax.set_yticks(TICKS)
    ax.xaxis.set_major_formatter(dollar_fmt)
    ax.yaxis.set_major_formatter(dollar_fmt)
    ax.set(xlim=ext[:2], ylim=ext[2:], xlabel=f"{xc} (log scale)", ylabel=f"{yc} (log scale)")
    ax.grid(False)
    ax.set_title(f"{yc} vs {xc}.  Lines: {yc} = {', '.join(f'{r:g}' for r in ratios)} x {xc}", fontsize=10)
axes[1, 1].set_xticks(np.log10([6e4, 8e4, 1e5, 1.2e5, 1.5e5]))      # MSA income spans < half a decade
rate_colorbar(fig, hb, axes)

Reading:

- Loan vs income: two effects. The lowest acceptance is for small loans (below ~100k) at any income, a horizontal band: loan size itself, largely the manufactured segment. On top of that, acceptance falls in the top-left corner, beyond ~5x income, and there the colour follows the diagonals: that is the loan-to-income ratio. The separate strips at 5k and 15k are mostly subordinate liens, which are accepted (see the lien x loan amount table below).
- Property vs income: mainly horizontal bands (the property level), with the high property / income corner a little lower.
- Loan vs property: the cells above the 0.95 line (LTV > 95%) at low values are the red ones. That is LTV, i.e. a ratio.
- Property vs MSA income: horizontal bands. What matters is the property level, not the property relative to the local income level.

The next test measures the same thing out-of-fold.

### 4b. Does a ratio add signal beyond its two columns?

For each candidate ratio, two models on the two columns, with and without the ratio:

- gradient boosting (sklearn's HistGradientBoosting with its defaults; it chooses its number of trees by early stopping on an internal 10% split), standing in for the tree models;
- logistic regression with the encoding chosen above (spline on log + flags) for every column.

Score: out-of-fold AUC. Verdict with the project's acceptance rule: keep if the paired mean gain over the 5 folds is >= 2 x its SE. Income is multiplied by 1,000 so that all parts are in dollars. A ratio with a missing or <= 0 part is NaN.

This tests each ratio next to its own two columns only. In the full model other features may already carry part of it, so the final say is the CV of the feature stage.

In [ ]:
RATIOS = {"loan / income": ("loan", "income"), "property / income": ("property", "income"),
          "loan / property": ("loan", "property"), "property / MSA income": ("property", "MSA income"),
          "income / MSA income": ("income", "MSA income")}
gbm = HistGradientBoostingClassifier(random_state=0)                     # random_state: its internal split


def spline_block(x, fit):
    return encode(x[fit], x, "spline")


rows = []
for name, (a, b) in RATIOS.items():
    xa, xb = dollars[a].to_numpy(), dollars[b].to_numpy()
    ratio = np.divide(xa, xb, out=np.full(len(xa), np.nan), where=(xa > 0) & (xb > 0))
    auc_pair = {"gbm": ([], []), "linear": ([], [])}
    for k in np.unique(fold):
        fit, val = fold != k, fold == k
        for i, X in enumerate([np.column_stack([xa, xb]), np.column_stack([xa, xb, ratio])]):
            auc_pair["gbm"][i].append(roc_auc_score(y[val], gbm.fit(X[fit], y[fit]).predict_proba(X[val])[:, 1]))
        L = np.column_stack([spline_block(xa, fit), spline_block(xb, fit)])
        for i, X in enumerate([L, np.column_stack([L, spline_block(ratio, fit)])]):
            p = linear.fit(X[fit], y[fit]).predict_proba(X[val])[:, 1]
            auc_pair["linear"][i].append(roc_auc_score(y[val], p))
    for model, (without, with_ratio) in auc_pair.items():
        d = np.array(with_ratio) - np.array(without)
        se = d.std(ddof=1) / np.sqrt(len(d))
        rows.append({"ratio": name, "model": model, "AUC, 2 columns": np.mean(without),
                     "AUC, + ratio": np.mean(with_ratio), "gain": d.mean(), "SE": se, "keep (gain >= 2 SE)": d.mean() >= 2 * se})
pd.DataFrame(rows).set_index(["ratio", "model"]).round(4)

- Linear model: every ratio passes. The three money ratios add +0.010 to +0.017 AUC (>= 7 SE): an additive spline model can't draw a diagonal boundary, so the ratio has to be given. The two "relative to the area" ratios add only ~+0.001.
- Gradient boosting: gains are tiny. Loan / property (+0.0023) and property / income (+0.0014) pass the rule. Loan / income does not (+0.0003), and property / MSA income even loses a little (-0.0010), as the maps suggested.

=> FE
- Linear model: **loan / income, property / income, loan / property** (computed LTV), each as a spline on log with the sentinel flags. The two area ratios are optional.
- Trees: **computed LTV and property / income**. The rest is left out unless the feature-stage CV says otherwise.

### 4c. Scan of the crossings of the 12 strongest features

Features: the strongest of the signal ranking, one per redundant group ("aus-1", DTI, LTV, property value, income, loan amount, loan term, construction method, credit score type), plus three that are weak alone but likely to act through others (loan type, occupancy, lien status). Geography is left to its target encoding, and protected attributes are not candidates for features.

For each of the 66 pairs, an out-of-fold logistic regression:

- additive: one-hot of each feature (numerics cut into 10 equal-count bins with the same mass-point rule as the V matrix, missing = its own level). Any shape for each feature, no interaction.
- crossed: the same + one-hot of every (level of a, level of b) cell.

Gain = deviance explained (%) of crossed - additive, i.e. what the pair explains together beyond its two separate effects. Both models get the same penalty (sklearn's default L2, C = 1), and the score is out-of-fold, so a crossing can't gain by memorising its small cells. 10 bins instead of 20 keep the crossed cells large (~840 rows per cell on average for a 10 x 10 crossing).

Check against a suspicion: several of these columns go missing together (01_eda), so part of a gain could just be "both missing". The "values only" variant leaves out every crossed cell where either level is missing.

In [ ]:
TOP = ["aus-1", "debt_to_income_ratio", "loan_to_value_ratio", "property_value", "income", "loan_amount", "loan_term",
       "construction_method", "applicant_credit_score_type", "loan_type", "occupancy_type", "lien_status"]
level = {c: (discretize(train[c], 10) if c in NUMERIC else train[c]).cat.codes.to_numpy().astype(np.int64) + 1
         for c in TOP}                                                    # 0 = missing


def onehot(code, n_levels):
    return sp.csr_matrix((np.ones(len(code)), (np.arange(len(code)), code)), shape=(len(code), n_levels))


def crossing_gain(a, b):
    la, lb = level[a], level[b]
    na, nb = la.max() + 1, lb.max() + 1
    additive = sp.hstack([onehot(la, na), onehot(lb, nb)]).tocsr()
    crossed = sp.hstack([additive, onehot(la * nb + lb, na * nb)]).tocsr()
    values_cell = np.where((la == 0) | (lb == 0), 0, la * nb + lb)          # 0 = no crossed cell
    values_only = sp.hstack([additive, onehot(values_cell, na * nb)[:, 1:]]).tocsr()
    dev = {"additive": [], "crossed": [], "values only": []}
    for k in np.unique(fold):
        fit, val = fold != k, fold == k
        for name, X in [("additive", additive), ("crossed", crossed), ("values only", values_only)]:
            p = LogisticRegression(max_iter=1000).fit(X[fit], y[fit]).predict_proba(X[val])[:, 1]
            dev[name].append(deviance_explained(p, k))
    gain = np.array(dev["crossed"]) - np.array(dev["additive"])
    return {"a": a, "b": b, "additive": np.mean(dev["additive"]), "gain": gain.mean(),
            "SE": gain.std(ddof=1) / np.sqrt(len(gain)),
            "gain, values only": np.mean(dev["values only"]) - np.mean(dev["additive"])}


pairs = [(a, b) for i, a in enumerate(TOP) for b in TOP[i + 1:]]
crossings = pd.DataFrame(Parallel(n_jobs=-1)(delayed(crossing_gain)(a, b) for a, b in pairs))
crossings.sort_values("gain", ascending=False).head(15).round(2)

In [ ]:
G = crossings.pivot(index="a", columns="b", values="gain").reindex(index=TOP, columns=TOP)
G = G.fillna(G.T)                                                          # symmetric, diagonal stays empty

fig, ax = plt.subplots(figsize=(8.5, 7))
im = ax.imshow(G, cmap=SEQ, vmin=0, vmax=G.max().max())
for i in range(len(TOP)):
    for j in range(len(TOP)):
        if i != j and G.iloc[i, j] >= 0.5:
            ax.text(j, i, f"{G.iloc[i, j]:.1f}", ha="center", va="center", fontsize=7,
                    color="white" if G.iloc[i, j] > 0.6 * G.max().max() else "#0b0b0b")
ax.set_xticks(range(len(TOP)), TOP, rotation=90)
ax.set_yticks(range(len(TOP)), TOP)
ax.grid(False)
ax.spines[:].set_visible(False)
fig.colorbar(im, ax=ax, shrink=0.7, label="gain in deviance explained (points)")
ax.set_title("What each pair explains beyond its two separate effects (values >= 0.5 written)")
fig.tight_layout()

Reading:

- Every gain in the top 15 is >= 10 SE, so these are not noise.
- The suspicion was mostly wrong. "Values only" keeps nearly all of each gain: the missing-level cells explain 0.26 to 0.36 points for the pairs from the missingness block (DTI, LTV, property value, loan term), and <= 0.07 elsewhere.
- Strongest: DTI x loan type (+2.7), DTI x LTV (+2.4), property value x loan term (+2.1), LTV x loan term (+2.0), LTV x property value (+1.8, i.e. the reported vs computed LTV relation), income x loan amount (+1.5, the ratio tested above), construction x loan type (+1.5), loan amount x lien status (+1.3).
- Loan term appears in 3 of the top 10 (5 of the top 15). Terms other than 360 months are mostly manufactured homes, so these crossings partly re-describe that segment.
- Weak alone does not mean useless: loan type (0.54 alone) is in 3 of the top 10 crossings.

### 4d. The strongest crossings, read as tables

Accept rate per cell, same colour scale as the maps above. Cells with < 50 rows are blank.

Bands: DTI keeps the field's own buckets, and the exact values 36 to 49 are merged into one band because 01_eda found them flat. LTV is cut at the usual underwriting thresholds (80 = no mortgage insurance needed, 90, 95, 100). Loan amount uses doubling bands (equal width on the log scale).

In [ ]:
def rate_grid(ax, a, b, title):
    # Accept rate per (a, b) cell; colour = log-odds vs overall, as in the maps; cells with < 50 rows blank.
    n = pd.crosstab(a, b, dropna=False)
    acc = pd.crosstab(a, b, values=train[TARGET], aggfunc="sum", dropna=False).fillna(0)
    show = n >= 50
    shade = (np.log((acc + 0.5) / (n - acc + 0.5)) - LOGIT_OVERALL).where(show)
    im = ax.imshow(shade, cmap=DIV.reversed(), vmin=-2, vmax=2, aspect="auto")
    for i in range(n.shape[0]):
        for j in range(n.shape[1]):
            if show.iloc[i, j]:
                ax.text(j, i, f"{acc.iloc[i, j] / n.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8,
                        color="white" if abs(shade.iloc[i, j]) > 1.2 else "#0b0b0b")
    ax.set_xticks(range(n.shape[1]), n.columns)
    ax.set_yticks(range(n.shape[0]), n.index)
    ax.set(xlabel=b.name, ylabel=a.name)
    ax.tick_params(length=0)
    ax.grid(False)
    ax.spines[:].set_visible(False)
    ax.set_title(title, fontsize=10)
    return im


def ordered(values, order, name):
    return pd.Series(pd.Categorical(values, categories=order, ordered=True), index=train.index, name=name)


DTI_BAND = {"<20%": "<20", "20%-<30%": "20-30", "30%-<36%": "30-36", "50%-60%": "50-60", ">60%": ">60",
            **{str(i): "36-49" for i in range(36, 50)}}
dti_band = ordered(train.debt_to_income_ratio.astype("string").map(DTI_BAND).fillna("missing"),
                   ["<20", "20-30", "30-36", "36-49", "50-60", ">60", "missing"], "DTI band")
loan_type = ordered(train.loan_type.astype("string").map({"1": "Conventional", "2": "FHA", "3": "VA", "4": "USDA"}),
                    ["Conventional", "FHA", "VA", "USDA"], "loan type")
ltv_band = ordered(pd.cut(train.loan_to_value_ratio, [0, 80, 90, 95, 100, np.inf],
                          labels=["<=80", "80-90", "90-95", "95-100", ">100"]).astype("string").fillna("missing"),
                   ["<=80", "80-90", "90-95", "95-100", ">100", "missing"], "LTV band")
construction = ordered(train.construction_method.astype("string").map({"1": "site-built", "2": "manufactured"}),
                       ["site-built", "manufactured"], "construction")
lien = ordered(train.lien_status.astype("string").map({"1": "first", "2": "subordinate"}),
               ["first", "subordinate"], "lien")
amount_band = ordered(pd.cut(train.loan_amount, [0, 25e3, 50e3, 100e3, 200e3, 400e3, np.inf],
                             labels=["<=25k", "25-50k", "50-100k", "100-200k", "200-400k", ">400k"]).astype("string"),
                      ["<=25k", "25-50k", "50-100k", "100-200k", "200-400k", ">400k"], "loan amount")

fig, axes = plt.subplots(2, 2, figsize=(12, 8.5), layout="constrained", height_ratios=[7, 2.6])
rate_grid(axes[0, 0], dti_band, loan_type, "DTI x loan type")
rate_grid(axes[0, 1], dti_band, ltv_band, "DTI x LTV")
rate_grid(axes[1, 0], construction, loan_type, "construction x loan type")
im = rate_grid(axes[1, 1], lien, amount_band, "lien x loan amount")
rate_colorbar(fig, im, axes)

Reading:

- DTI x loan type: the DTI limit depends on the program. At 50-60%, conventional loans are accepted 0.33, FHA 0.89, VA 0.92. Above 60%: conventional 0.06, FHA 0.02, VA still 0.61. This seems to match the programs' published rules: conventional (Fannie Mae DU) up to 50%, FHA up to ~57%, VA with no hard cap (residual income instead). To verify against the program guides.
- DTI x LTV: still +1.7 points after DTI x loan type is in the model (checked separately, SE 0.06), so it is not only the program effect again. High-DTI files do better at LTV 95-100 than at <= 80 (0.87 against 0.57 at DTI 50-60). A missing LTV is bad at every known DTI (0.27 to 0.58, worst at DTI < 20), but not when DTI is missing too (0.79: mostly the entity applicants).
- Construction x loan type: manufactured homes are denied almost only as conventional loans: 0.31 on 8,487 rows (89% of the manufactured segment), against 0.86 as FHA and 0.84 as VA. Site-built homes are at 0.87 to 0.91 in every program.
- Lien x loan amount: opposite directions. First liens rise from 0.47 (<= 25k) to 0.91 (> 400k) with the amount, while subordinate liens fall from 0.91 to 0.60-0.62.

=> FE
- Linear model: crossed features **DTI band x loan type**, **DTI band x LTV band**, **construction x loan type**, **lien x loan amount band** (or one loan-amount spline per lien status). Then the other scanned pairs with a gain >= 1 point, one at a time, CV deciding.
- All models: a **"manufactured + conventional"** flag (8,487 rows, 0.31), and a candidate **"DTI above the program's limit"** flag (conventional > 50%, FHA and VA > 60%; USDA has too few rows above 50% to tell and is grouped with conventional). These limits were read on the training set, so their gain must be confirmed by the feature-stage CV.
- Trees find these crossings themselves. For them the explicit flags are cheap shortcuts, and the CV decides whether they help.

## 5. PCA: is there low-dimensional structure?

Three questions: how many directions does the data really have, what are the main ones, and does the target line up with them (i.e. would PCA components make good features)?

The matrix is the one a linear model would see:

- Protected attributes are left out: they are not feature candidates, and components mixing race with finance would be useless for the fair model.
- Geography IDs (county, MSA, state) are left out: their one-hot columns would swamp everything, and they go to target encoding. The census numerics stay.
- Numerics: rank among the positive values (0 to 1, mid-rank for ties), with the sentinel flags (missing, <= 0) as separate columns. Ranks make every numeric column robust to the extreme values (LTV up to $6 \cdot 10^8$) with a single rule.
- Categoricals: one-hot, levels under 1% of rows merged into "rare" (the same 1% rule as the tree leaves of the signal ranking). Missing is a level.
- Every column is standardized, so each one counts the same in the variance.

PCA is fitted on all training rows. It uses no labels, so this leaks nothing into the out-of-fold scores below.

In [ ]:
GEO_IDS = ["county_code", "derived_msa-md", "state_code"]
RARE = 0.01                                                            # see text above


def rank01(x):
    # Rank among the positive values, 0 to 1; missing and <= 0 get 0.5 and are carried by their flags.
    pos = np.sort(x[x > 0])
    r = (np.searchsorted(pos, x, "left") + np.searchsorted(pos, x, "right")) / (2 * len(pos))
    return np.where(np.isnan(x) | (x <= 0), 0.5, r)


cols = {}
for c in NUM:
    x = train[c].to_numpy()
    cols[c], cols[f"{c} missing"], cols[f"{c} <= 0"] = rank01(x), np.isnan(x), x <= 0
for c in [c for c in FEATURES if c not in NUM + PROTECTED + GEO_IDS]:
    s = train[c].astype("string").fillna("missing")
    s = s.where(s.map(s.value_counts(normalize=True)) >= RARE, "rare")
    for lev in s.unique():
        cols[f"{c} = {lev}"] = (s == lev).to_numpy()
design = pd.DataFrame(cols).astype(float)
design = design.loc[:, design.std() > 0]                               # drop flags that never fire

Z = StandardScaler().fit_transform(design)
pca = PCA().fit(Z)
scores = pca.transform(Z)
evr = pca.explained_variance_ratio_
need = {t: int(np.searchsorted(evr.cumsum(), t)) + 1 for t in (0.5, 0.8, 0.9)}

fig, ax = plt.subplots(figsize=(9, 3.8))
k = np.arange(1, 41)
ax.bar(k, evr[:40], color=BLUE, width=0.7, label="each component")
ax.plot(k, evr.cumsum()[:40], "o-", color=INK, ms=3, lw=1, label="cumulative")
ax.set(xlabel="component", ylabel="share of variance", ylim=(0, 1), xlim=(0.3, 40.7))
ax.legend(loc="center right")
ax.set_title(f"{design.shape[1]} columns: {need[0.5]} components for 50% of the variance, "
             f"{need[0.8]} for 80%, {need[0.9]} for 90%")
fig.tight_layout()

The data is not compressible. After the first two components (17% and 8%), each one adds at most 5%, from the 9th on less than 2%, and 50% of the variance needs 11 components. Most columns carry their own information, as the V matrix already suggested (few groups with V >= 0.7).

### 5b. What are the main components?

For each of the first 6 components: its strongest loadings at both ends, and who sits in its lowest and highest deciles. The segments are the row states and groups found earlier, in 01_eda and in this notebook.

In [ ]:
YES_NO = ["reverse_mortgage", "open-end_line_of_credit", "business_or_commercial_purpose", "negative_amortization",
          "interest_only_payment", "balloon_payment", "other_nonamortizing_features"]
SEGMENTS = pd.DataFrame({
    "accepted": y == 1,
    "exempt (1111)": EXEMPT,
    "entity applicant": train.applicant_sex.astype("string") == "4",
    "manufactured": train.construction_method.astype("string") == "2",
    "no AUS": train["aus-1"].astype("string") == "6",
    "census zeros": train.tract_population == 0,
    "subordinate lien": train.lien_status.astype("string") == "2",
    "FHA / VA / USDA": train.loan_type.astype("string") != "1",
    "a yes loan flag": (train[YES_NO].astype("string") == "1").any(axis=1),
})

ends, profile = {}, {}
for i in range(6):
    pc = f"PC{i + 1}"
    load = pd.Series(pca.components_[i], index=design.columns).sort_values()
    ends[pc] = {"variance %": 100 * evr[i],
                "- end": ", ".join(f"{c} ({v:+.2f})" for c, v in load.head(3).items()),
                "+ end": ", ".join(f"{c} ({v:+.2f})" for c, v in load.tail(3)[::-1].items())}
    decile = pd.qcut(scores[:, i], 10, labels=False)
    profile[f"{pc} lowest 10%"] = SEGMENTS[decile == 0].mean()
    profile[f"{pc} highest 10%"] = SEGMENTS[decile == 9].mean()
display(pd.DataFrame(ends).T.round(1))
pd.DataFrame(profile).T.round(2)

The main components are the segments found before, rediscovered without any label:

- PC1 (17%), standard vs non-standard file: from the plain loan (every yes/no loan flag = "no") to exempt "1111" records, "yes" loan flags, no AUS and manufactured homes. Its highest decile is accepted 0.65 (29% exempt, 69% no AUS).
- PC2 (8%), manufactured homes: its highest decile is 91% manufactured, accepted 0.39.
- PC3 (4.8%), business / investment / entity applicants: business purpose, investment property, income missing.
- PC4 (3.6%), census zeros, i.e. location unknown, opposed to manufactured / subordinate liens.
- PC5 (3.5%), loan program: conventional vs FHA.
- PC6 (2.9%), subordinate liens and open-end lines of credit.

(Read the exact shares in the tables above.)

### 5c. The first plane: where are the denials?

Left: how many rows (log colour scale). Right: accept rate, same colour scale as the interaction maps (hexagons with < 50 rows blank). The axes cover the central 99.8% of each component.

In [ ]:
pc1, pc2 = scores[:, 0], scores[:, 1]
ext = [*np.quantile(pc1, [0.001, 0.999]), *np.quantile(pc2, [0.001, 0.999])]
fig, (a, b) = plt.subplots(1, 2, figsize=(13, 5.2), layout="constrained")
h1 = a.hexbin(pc1, pc2, gridsize=45, extent=ext, bins="log", mincnt=1, cmap=SEQ, linewidths=0.2, edgecolors=SURFACE)
fig.colorbar(h1, ax=a, shrink=0.8, label="rows (log scale)")
h2 = b.hexbin(pc1, pc2, C=y, reduce_C_function=rel_logit, gridsize=45, extent=ext, mincnt=50,
              cmap=DIV.reversed(), vmin=-2, vmax=2, linewidths=0.2, edgecolors=SURFACE)
rate_colorbar(fig, h2, b)
for ax, title in [(a, "where the rows are"), (b, "accept rate")]:
    ax.set(xlabel=f"PC1 ({100 * evr[0]:.0f}%): standard -> non-standard file",
           ylabel=f"PC2 ({100 * evr[1]:.0f}%): -> manufactured", xlim=ext[:2], ylim=ext[2:])
    ax.grid(False)
    ax.set_title(title)

The plane shows separate clusters rather than one cloud, because the leading components are driven by blocks of indicators that fire together:

- bottom left, the main cloud: site-built homes, accepted ~0.90 on average, with both red and blue cells inside (the first two components can't separate them);
- PC2 ~5 to 11: manufactured homes, mostly red;
- PC2 > 12: a second, smaller manufactured cluster, almost entirely denied;
- far right (PC1 ~27): the exempt "1111" records, pushed far out because 14 indicators fire together, and accepted. So the "non-standard" end of PC1 mixes accepted exempt records with denied no-AUS / yes-flag files, which is why its highest decile is only at 0.65.

What makes the small cluster? Comparing its rows column by column with the other manufactured rows, one difference stands out: zero census fields (location unknown). The crossing:

In [ ]:
def home_and_location(d):
    home = d.construction_method.astype("string").map({"1": "site-built", "2": "manufactured"})
    census = pd.Series(np.where(d.tract_population == 0, "census zeros", "census present"), index=d.index)
    return home + " / " + census


seg = train.groupby(home_and_location(train))[TARGET].agg(["size", "mean"])
seg.columns = ["train rows", "accept rate"]
seg["test rows"] = home_and_location(test).value_counts()
seg.round(3)

- Manufactured homes with an unknown location: 1,516 rows, accepted 0.016 (653 such rows in test). Neither part explains it alone: site-built homes with an unknown location are accepted 0.56, manufactured homes with a known location 0.44.
- 72% of all location-unknown rows are manufactured homes.
- Hypothesis: for a manufactured home, an unknown location means the home has no site yet. That is a state of the file (process), not a property of the borrower, like the "preapproval" rule.

=> FE: a **"manufactured + location unknown"** flag. It is nearly deterministic, so it also goes to the leakage scan and gets a leakage verdict, like "preapproval". Trees find it in two splits, but the linear model needs it explicitly.

### 5d. Would PCA components make good features?

Out-of-fold AUC of a logistic regression on the first k components, for growing k. Keeping all components is the same model as a logistic regression on the whole matrix: the L2 penalty does not change under a rotation. For reference, the LightGBM baseline of "results/runs.csv" (all raw features, including geography and protected attributes).

In [ ]:
runs = pd.read_csv(Path.cwd().parent / "results" / "runs.csv")
LGBM_BASELINE = runs.loc[runs.experiment.str.contains("lgbm_baseline"), "mean_auc"].iloc[-1]

K = [1, 2, 3, 5, 10, 20, 40, scores.shape[1]]
pc_auc = {}
for n_pc in K:
    fold_scores = []
    for k in np.unique(fold):
        fit, val = fold != k, fold == k
        m = LogisticRegression(max_iter=2000).fit(scores[fit, :n_pc], y[fit])    # max_iter: convergence only
        fold_scores.append(roc_auc_score(y[val], m.predict_proba(scores[val, :n_pc])[:, 1]))
    pc_auc[n_pc] = np.mean(fold_scores)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(K, list(pc_auc.values()), "o-", color=BLUE, lw=2, ms=6)
for n_pc, v in pc_auc.items():
    ax.annotate(f"{v:.3f}", (n_pc, v), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8)
ax.axhline(LGBM_BASELINE, color=MUTED, lw=1)
ax.text(K[0], LGBM_BASELINE, f"LightGBM baseline {LGBM_BASELINE:.3f}", va="bottom", fontsize=8, color=INK)
ax.set_xscale("log")
ax.set_xticks(K, [str(v) for v in K])
ax.minorticks_off()
ax.set(xlabel="number of components kept (log scale)", ylabel="out-of-fold AUC", ylim=(0.70, 0.93))
ax.set_title("Logistic regression on the first k components")
fig.tight_layout()

- The target does not line up with the main components: 10 components reach 0.80, 40 reach 0.87, and only all of them reach 0.89. The signal is spread over many small directions, so compressing throws it away.
- Side result: a plain logistic regression on this matrix, with no feature engineering from this notebook (no crossings, no splines, no geography, no protected attributes), already reaches 0.89, only ~0.013 below the LightGBM baseline. The linear model is a serious candidate, both for the ensemble and as the interpretable fair model.

=> FE
- **No PCA components as features.** They lose signal, and the few leading ones are the segments already turned into explicit flags above (exempt, manufactured, entity, location unknown, program, subordinate lien). A named flag is also easier to explain in the fairness report than "PC3".
- The one summary PCA does suggest, PC1 = "how non-standard is this file", is already covered by those flags. Low priority, only if a single "non-standard file" score is wanted.

## 6. Segments: where do the denials concentrate, and does the plan miss any?

"reports/feature_plan.md" collects every feature-engineering decision of 01_eda and this notebook. This section checks it against the data from three angles:

- a depth-3 tree gives readable segments, each compared with the plan's flags;
- the same tree on the rows no plan flag covers: where are the remaining denials?
- clustering (KMeans) and a t-SNE map: is there structure the plan does not name?

The tree sees the columns as the tree models will get them in the plan: no protected attributes, the two derived duplicates dropped, geography IDs left to their encoding. Numerics are raw (the tree handles NaN itself) and categoricals one-hot (levels < 1% merged, as for the PCA). Depth 3 gives at most 8 segments, each >= 1% of the rows (the same leaf rule as before). The tree is fitted on all training rows because it describes; only the depth curve below is scored, out-of-fold.

In [ ]:
def as_str(c):
    return train[c].astype("string")


DERIVED = ("derived_loan_product_type", "derived_dwelling_category")
tree_X = pd.concat([train[NUM], design[[c for c in design.columns if " = " in c and not c.startswith(DERIVED)]]], axis=1)

# The row-state and segment flags of "reports/feature_plan.md".
manufactured = as_str("construction_method").isin(["2"]).to_numpy(bool)
location_unknown = (train.tract_population == 0).to_numpy()
dti = as_str("debt_to_income_ratio")
dti_limit = np.where(as_str("loan_type").isin(["2", "3"]).to_numpy(bool),                       # FHA, VA: 60%
                     dti.isin([">60%"]).to_numpy(bool), dti.isin(["50%-60%", ">60%"]).to_numpy(bool))  # others: 50%
PLAN_FLAGS = pd.DataFrame({
    "exempt record": EXEMPT,
    "entity applicant": as_str("applicant_sex").isin(["4"]).to_numpy(bool),
    "location unknown": location_unknown,
    "income <= 0": (train.income <= 0).to_numpy(),
    "LTV missing": train.loan_to_value_ratio.isna().to_numpy(),
    "manufactured + conventional": manufactured & as_str("loan_type").isin(["1"]).to_numpy(bool),
    "manufactured + location unknown": manufactured & location_unknown,
    "DTI above program limit": dti_limit,
})
flag_table = pd.DataFrame({"rows": PLAN_FLAGS.sum(), "accept rate": [y[PLAN_FLAGS[c]].mean() for c in PLAN_FLAGS]})
LOW_FLAGS = flag_table.index[flag_table["accept rate"] < OVERALL]
covered = PLAN_FLAGS[LOW_FLAGS].any(axis=1).to_numpy()            # row carries a low-acceptance plan flag
denials = (1 - y).sum()
print(f"rows with a low-acceptance plan flag: {covered.mean():.1%}, accepted {y[covered].mean():.3f}, "
      f"holding {(1 - y[covered]).sum() / denials:.0%} of all denials; the other rows are accepted {y[~covered].mean():.3f}")
flag_table.round(3)

In [ ]:
depth_auc = {}
for depth in [1, 2, 3, 4, 6, 8]:
    p = np.empty(len(y))
    for k in np.unique(fold):
        fit, val = fold != k, fold == k
        m = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=LEAF_SHARE, random_state=0).fit(tree_X[fit], y[fit])
        p[val] = m.predict_proba(tree_X[val])[:, 1]
    depth_auc[depth] = fold_auc(p).mean()
pd.Series(depth_auc, name="out-of-fold AUC").rename_axis("tree depth").to_frame().T.round(3)

In [ ]:
def leaf_table(tree, X, rows):
    # One row per leaf: its rule, size, accept rate, share of ALL training denials, and plan coverage.
    t, names = tree.tree_, X.columns
    rules = {}

    def walk(node, conds):
        if t.children_left[node] == -1:
            rules[node] = " & ".join(conds)
            return
        name, thr = names[t.feature[node]], t.threshold[node]
        if " = " in name:                                          # one-hot column: left branch = "not this level"
            left, right = name.replace(" = ", " != "), name
        else:
            left, right = f"{name} <= {thr:,.1f}", f"{name} > {thr:,.1f}"
            if X[name].isna().any():
                if t.missing_go_to_left[node]:
                    left += " or missing"
                else:
                    right += " or missing"
        walk(t.children_left[node], conds + [left])
        walk(t.children_right[node], conds + [right])

    walk(0, [])
    d = pd.DataFrame({"leaf": tree.apply(X), "y": y[rows], "covered": covered[rows]})
    g = d.groupby("leaf")
    out = pd.DataFrame({"segment": pd.Series(rules), "rows": g.size(), "accept rate": g.y.mean(),
                        "share of all denials": g.y.apply(lambda v: (1 - v).sum()) / denials,
                        "with a plan flag": g.covered.mean(),
                        "accept rate without plan flag": d[~d.covered].groupby("leaf").y.mean()})
    return out.dropna(subset=["rows"]).sort_values("accept rate").reset_index(drop=True)


seg_tree = DecisionTreeClassifier(max_depth=3, min_samples_leaf=LEAF_SHARE, random_state=0).fit(tree_X, y)
segments = leaf_table(seg_tree, tree_X, np.ones(len(y), bool))
segments.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6.5))
pos = np.arange(len(segments))
share_rows = segments["rows"] / len(y)
ax.barh(pos - 0.2, share_rows, height=0.38, color=BLUE, label="share of all rows")
ax.barh(pos + 0.2, segments["share of all denials"], height=0.38, color=ORANGE, label="share of all denials")
for p_, a, b in zip(pos, share_rows, segments["share of all denials"]):
    ax.text(a, p_ - 0.2, f" {a:.1%}", va="center", fontsize=7, color=INK)
    ax.text(b, p_ + 0.2, f" {b:.1%}", va="center", fontsize=7, color=INK)
labels = [textwrap.fill(f"{r}   => accepted {a:.2f}", 70) for r, a in zip(segments.segment, segments["accept rate"])]
ax.set_yticks(pos, labels, fontsize=7.5)
ax.invert_yaxis()
ax.grid(axis="y", visible=False)
ax.set_xlim(0, 1)
ax.set_xlabel("share")
ax.legend(loc="lower right")
ax.set_title("Depth-3 tree: each segment's share of the rows vs its share of the denials")
fig.tight_layout()

Reading:

- Denials are concentrated: the small segments together hold ~13% of the rows and ~60% of the denials. The large "everything else" segment is ~87% of the rows, accepted 0.93, and still holds ~40% of the denials, spread thinly.
- The tree's first split is DTI > 60%, then manufactured homes, then the process fields ("initially_payable_to_institution" = 3, "aus-1" = 6).
- Every low-acceptance segment is almost entirely covered by a plan flag (manufactured + conventional, DTI above the program limit), except one: "initially_payable_to_institution" = 3, accepted 0.000 whether or not a plan flag is present. It is in the plan as a raw column waiting for its leakage verdict, not as a flag.
- Depth curve: one tree of depth 3 already reaches 0.79 AUC, depth 8 reaches 0.87. Most of the ranking comes from a few coarse segments; the rest is fine detail.

### 6b. Where are the denials that no plan flag covers?

The same depth-3 tree, fitted only on the rows without any low-acceptance plan flag. If the plan missed a segment, it should show up here as a leaf with a low accept rate.

In [ ]:
rest = ~covered
rest_tree = DecisionTreeClassifier(max_depth=3, min_samples_leaf=LEAF_SHARE, random_state=0).fit(tree_X[rest], y[rest])
print(f"{rest.sum():,} rows without a plan flag, accepted {y[rest].mean():.3f}, "
      f"holding {(1 - y[rest]).sum() / denials:.0%} of all denials")
leaf_table(rest_tree, tree_X[rest], rest).drop(columns=["with a plan flag", "accept rate without plan flag"]).round(3)

- The only sharp segment left is again "initially_payable_to_institution" = 3: ~1,070 rows, none accepted, about a fifth of the remaining denials.
- Every other leaf is accepted ~0.80 or more (lowest 0.798). The next splits are no AUS ("aus-1" = 6) and LTV above ~103, both raw columns the models already get.
- So the remaining denials are diffuse. They are not a missing segment. They need the fine-grained part of the model (DTI detail, geography, credit score type), not a new flag.

### 6c. Clustering and t-SNE: structure without labels

KMeans on the standardized PCA matrix for k = 2 to 10 (5 restarts each, best kept, to avoid a poor local optimum). Silhouette measures how well separated the clusters are (1 = perfectly, ~0 = arbitrary cuts of one cloud). It costs $O(n^2)$, so it is computed on a random sample of 10,000 rows.

In [ ]:
SAMPLE = np.random.default_rng(0).choice(len(Z), 10_000, replace=False)   # seed 0: arbitrary, fixed
kmeans, silhouette = {}, {}
for n_clusters in range(2, 11):
    kmeans[n_clusters] = KMeans(n_clusters=n_clusters, n_init=5, random_state=0).fit(Z)
    silhouette[n_clusters] = silhouette_score(Z[SAMPLE], kmeans[n_clusters].labels_[SAMPLE])

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(list(silhouette), list(silhouette.values()), "o-", color=BLUE, lw=2, ms=6)
ax.set(xlabel="number of clusters k", ylabel="silhouette (10k-row sample)", ylim=(0, 0.65))
ax.set_title("How separated are the KMeans clusters?")
fig.tight_layout()
plt.show()

k3 = kmeans[3].labels_
profile = SEGMENTS.groupby(k3).mean()
profile.insert(0, "rows", np.bincount(k3))
profile.rename_axis("cluster (k = 3)").round(2)

Only two or three real clusters exist. Silhouette is ~0.56 at k = 2 (the exempt "1111" records vs everything else) and ~0.32 at k = 3 (plus manufactured homes), then falls to ~0.1 from k = 4 on: more clusters just cut the main cloud arbitrarily. Both real clusters are in the plan already (exempt record; manufactured + conventional and manufactured + location unknown).

t-SNE draws the same data as a 2-D map where close points are similar rows. It runs on 10,000 rows (the same sample) and on the first 50 PCA components, the usual step before t-SNE to remove noise. sklearn defaults otherwise (perplexity 30), seed fixed. Distances between islands, and island sizes, mean nothing in t-SNE: only "which points sit together" does.

Left: outcome. Right: whether the row carries a low-acceptance plan flag, or the process rule "initially_payable_to_institution" = 3.

In [ ]:
emb = TSNE(random_state=0, n_jobs=-1).fit_transform(scores[SAMPLE, :50])
yy, cov = y[SAMPLE], covered[SAMPLE]
payable3 = as_str("initially_payable_to_institution").isin(["3"]).to_numpy(bool)[SAMPLE]
print(f"the first 50 components carry {evr[:50].sum():.0%} of the variance")

fig, (a, b) = plt.subplots(1, 2, figsize=(13, 6.2), layout="constrained")
a.scatter(*emb[yy == 1].T, s=3, color="#c3c2b7", lw=0, label="accepted")
a.scatter(*emb[yy == 0].T, s=4, color=ORANGE, lw=0, label="denied")
a.set_title("outcome")
b.scatter(*emb[~cov & ~payable3].T, s=3, color="#c3c2b7", lw=0, label="no plan flag")
b.scatter(*emb[cov].T, s=4, color=BLUE, lw=0, label="low-acceptance plan flag")
b.scatter(*emb[payable3 & ~cov].T, s=4, color=AQUA, lw=0, label="initially payable = 3, no flag")
b.set_title("plan coverage")
for ax in (a, b):
    ax.set_axis_off()
    ax.legend(loc="lower left", markerscale=4)

Reading: the map is a set of islands, because most columns are one-hot and every island is a combination of levels. The dense orange islands on the left are blue or aqua on the right: the concentrations of denials are covered by a plan flag or by the process rule. What stays grey on the right carries the scattered denials found above.

=> FE, from the segment analysis
- The plan covers every denial segment the data shows. No new flag is needed.
- One addition, now in the plan: **"initially_payable_to_institution" = 3** named explicitly in the leakage list (it is a raw column already). It is the only sharp segment outside the plan's flags, and the fair / clean model must know it by name to exclude it.
- **No cluster id as a feature**: the only real clusters are exempt records and manufactured homes, both already flags.